In [1]:
from __future__ import annotations

from pathlib import Path
import shutil
import pandas as pd

RANDOM_STATE = 42
BANGKOK_NAME = "กรุงเทพมหานคร"
BANGKOK_TRAIN_LIMIT = 2000

def first_existing_path(candidates: list[Path]) -> Path:
    for p in candidates:
        if p.exists():
            return p
    pretty = "\n".join(f"- {c.resolve()}" for c in candidates)
    raise FileNotFoundError(f"Could not find any of these paths:\n{pretty}")

# --- Locate input files/folders (works whether cwd is repo root or data_preprocessor/)
labels_csv = first_existing_path([
    Path("data/plate_lower/labels.csv"),
    Path("../data/plate_lower/labels.csv"),
])
images_dir = first_existing_path([
    Path("data/plate_lower/data_resized_128x32"),
    Path("../data/plate_lower/data_resized_128x32"),
])

output_root = first_existing_path([
    Path("data/plate_lower"),
    Path("../data/plate_lower"),
])
train_dir = output_root / "lower_train"
test_dir = output_root / "lower_test"
train_dir.mkdir(parents=True, exist_ok=True)
test_dir.mkdir(parents=True, exist_ok=True)

print("labels_csv:", labels_csv.resolve())
print("images_dir:", images_dir.resolve())
print("train_dir:", train_dir.resolve())
print("test_dir:", test_dir.resolve())

# --- Load labels
df = pd.read_csv(labels_csv)
if "filename" not in df.columns:
    raise ValueError("labels.csv must contain a 'filename' column")

province_col = "province_description" if "province_description" in df.columns else "label"
df[province_col] = df[province_col].astype(str).str.strip()

# --- Split rules
bangkok_mask = df[province_col] == BANGKOK_NAME
df_bkk = df[bangkok_mask].copy()
df_other = df[~bangkok_mask].copy()

# Bangkok: first 2000 to train, rest to test (shuffle for randomness but deterministic)
if len(df_bkk) > 0:
    df_bkk = df_bkk.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
bkk_train_n = min(BANGKOK_TRAIN_LIMIT, len(df_bkk))
df_bkk_train = df_bkk.iloc[:bkk_train_n].copy()
df_bkk_test = df_bkk.iloc[bkk_train_n:].copy()

# Other provinces: 80/20 split per province
train_parts = [df_bkk_train]
test_parts = [df_bkk_test]

for province_name, g in df_other.groupby(province_col, sort=False):
    g = g.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    n = len(g)
    if n <= 1:
        # Can't do 80/20 meaningfully; keep in train to avoid empty-label edge cases
        g_train = g
        g_test = g.iloc[0:0]
    else:
        n_train = int(round(n * 0.8))
        # Ensure both sides non-empty when possible
        n_train = max(1, min(n - 1, n_train))
        g_train = g.iloc[:n_train]
        g_test = g.iloc[n_train:]
    train_parts.append(g_train)
    test_parts.append(g_test)

df_train = pd.concat(train_parts, ignore_index=True)
df_test = pd.concat(test_parts, ignore_index=True)

# --- Copy images & write labels.csv (only keep rows with existing files)
def copy_and_filter(df_split: pd.DataFrame, dst_dir: Path) -> pd.DataFrame:
    kept_rows = []
    missing = 0
    for _, row in df_split.iterrows():
        filename = str(row["filename"]).strip()
        src = images_dir / filename
        if not src.exists():
            missing += 1
            continue
        dst = dst_dir / filename
        shutil.copy2(src, dst)
        kept_rows.append(row)
    out_df = pd.DataFrame(kept_rows).reset_index(drop=True)
    print(f"{dst_dir.name}: kept {len(out_df):,} rows, missing files {missing:,}")
    return out_df

df_train_out = copy_and_filter(df_train, train_dir)
df_test_out = copy_and_filter(df_test, test_dir)

train_labels_path = train_dir / "labels.csv"
test_labels_path = test_dir / "labels.csv"
df_train_out.to_csv(train_labels_path, index=False, encoding="utf-8-sig")
df_test_out.to_csv(test_labels_path, index=False, encoding="utf-8-sig")

print("Saved:", train_labels_path.resolve())
print("Saved:", test_labels_path.resolve())

# Quick sanity checks
overlap = set(df_train_out["filename"]) & set(df_test_out["filename"])
print("overlap filenames:", len(overlap))
print("Bangkok train/test:", (df_train_out[province_col] == BANGKOK_NAME).sum(), "/", (df_test_out[province_col] == BANGKOK_NAME).sum())

labels_csv: C:\Users\Tanaphat\Desktop\Coding\ALPR\data\plate_lower\labels.csv
images_dir: C:\Users\Tanaphat\Desktop\Coding\ALPR\data\plate_lower\data_resized_128x32
train_dir: C:\Users\Tanaphat\Desktop\Coding\ALPR\data\plate_lower\lower_train
test_dir: C:\Users\Tanaphat\Desktop\Coding\ALPR\data\plate_lower\lower_test
lower_train: kept 3,130 rows, missing files 0
lower_test: kept 7,078 rows, missing files 0
Saved: C:\Users\Tanaphat\Desktop\Coding\ALPR\data\plate_lower\lower_train\labels.csv
Saved: C:\Users\Tanaphat\Desktop\Coding\ALPR\data\plate_lower\lower_test\labels.csv
overlap filenames: 0
Bangkok train/test: 2000 / 6791
